In [1]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
import pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

load_dotenv()

/opt/anaconda3/envs/langchain_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
fw = load_dataset(
    "HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True
)

In [3]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [4]:
fw.features

{'text': Value('string'),
 'id': Value('string'),
 'dump': Value('string'),
 'url': Value('string'),
 'date': Value('string'),
 'file_path': Value('string'),
 'language': Value('string'),
 'language_score': Value('float64'),
 'token_count': Value('int64')}

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [6]:
pc = Pinecone(
    api_key=os.environ.get("PINECONE_API_KEY"),
    environment=os.environ.get("PINECONE_ENV"),
)

In [7]:
pc.list_indexes()

[
    {
        "name": "index-1536-dotproduct",
        "metric": "dotproduct",
        "host": "index-1536-dotproduct-h2xw5gc.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "example-index",
        "metric": "cosine",
        "host": "example-index-h2xw5gc.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
         

In [8]:
print("Model dimension:", model.get_sentence_embedding_dimension())

pc.create_index(
    name="text",
    dimension=model.get_sentence_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

Model dimension: 384


{
    "name": "text",
    "metric": "cosine",
    "host": "text-h2xw5gc.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
          

In [9]:
index = pc.Index("text")

In [10]:
# Define the number of items to process (subset size)
subset_size = 10000

# Iterate over the dataset and prepare data for upsert
vectors_to_upsert = []
for i, item in enumerate(fw):
    if i >= subset_size:
        break

    text = item["text"]
    unique_id = str(item["id"])
    language = item["language"]

    # Create an embedding for the text
    embedding = model.encode(text, show_progress_bar=True).tolist()

    # Prepare metadata
    metadata = {"language": language}

    # Append the vector data to the list
    vectors_to_upsert.append((unique_id, embedding, metadata))

# Upsert the vectors into the Pinecone index in batches
batch_size = 1000
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i : i + batch_size]
    index.upsert(vectors=batch)

print(f"Upserted {len(vectors_to_upsert)} vectors into the Pinecone index.")

Batches: 100%|██████████| 1/1 [00:00<00:00, 100.94it/s]


Upserted 10000 vectors into the Pinecone index.
